# LRK10L ML model

## Setup

### Imports & installations

In [ ]:
%load_ext autoreload
%autoreload 2

from lrk10l.evaluation import LOGOEvaluator
from lrk10l.pu import run_pu_bagging, PUFilterMixin

import pyhmmer
from pyhmmer.easel import MSAFile, Alphabet
from Bio import SeqIO
import esm
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, accuracy_score, average_precision_score, precision_recall_curve, log_loss, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import hypergeom, friedmanchisquare, studentized_range
import scikit_posthocs as sp
import torch
from pathlib import Path
import os
import itertools
from itertools import combinations
import math
from functools import reduce
from collections import Counter

### Helper functions

In [ ]:
def clean_gene_id(gene_id: str) -> str:
    return gene_id.split('|')[0]

def get_scale_pos_weight(y):
    return (y == 0).sum() / (y == 1).sum()

### Load HMMs

In [ ]:
os.makedirs('../data/hmms', exist_ok=True)

!wget -O ../data/hmms/PF00069.hmm.gz "https://www.ebi.ac.uk/interpro/wwwapi/entry/pfam/PF00069?annotation=hmm"
!gunzip -f ../data/hmms/PF00069.hmm.gz
!file ../data/hmms/PF00069.hmm

!wget -O ../data/hmms/PF07714.hmm.gz "https://www.ebi.ac.uk/interpro/wwwapi/entry/pfam/PF07714?annotation=hmm"
!gunzip -f ../data/hmms/PF07714.hmm.gz
!file ../data/hmms/PF07714.hmm

In [ ]:
def read_hmms(hmm_dir: Path) -> list[pyhmmer.plan7.HMM]:
    hmms = []
    for hmm_file in hmm_dir.glob('*.hmm'):
        with pyhmmer.plan7.HMMFile(hmm_file) as f:
            hmms.append(f.read())
    return hmms

def search_proteome_for_kinases(proteome_path: Path, hmms: list[pyhmmer.plan7.HMM]) -> set[str]:
    with pyhmmer.easel.SequenceFile(proteome_path, digital=True) as f:
        sequences = f.read_block()

    unique_names = set()
    for hmm in hmms:
        try:
            pipeline = pyhmmer.plan7.Pipeline(hmm.alphabet, bit_cutoffs='gathering')
            hits = pipeline.search_hmm(hmm, sequences)
        except pyhmmer.errors.MissingCutoffs:
            pipeline = pyhmmer.plan7.Pipeline(hmm.alphabet, E=0.001)
            hits = pipeline.search_hmm(hmm, sequences)
        except pyhmmer.errors.AlphabetMismatch:
            print(f'{proteome_path} is not protein, skipped.')
            continue

        for hit in hits.included:
            unique_names.add(clean_gene_id(hit.name))

    return unique_names

hmm_dir = Path('../data/hmms')
hmms = read_hmms(hmm_dir)
print(f'{len(hmms)} HMM(s) loaded')

## Load data

### Load sequences

In [ ]:
records = []
record_species = {}

for fname in os.listdir('../data/sequences'):
    species_name = os.path.splitext(fname)[0]

    filepath = os.path.join('../data/sequences', fname)
    with open(filepath) as handle:
        for rec in SeqIO.parse(handle, 'fasta'):
            rec.id = clean_gene_id(rec.id)
            rec.description = ''
            rec.seq = rec.seq.rstrip('*')
            records.append(rec)
            record_species[rec.id] = species_name

print(len(records), 'FASTA records')
print(len(set(record_species.values())), 'unique species')

### Load LRK10L gene IDs

In [ ]:
with open('../data/list.txt') as f:
    gene_ids = f.read().strip().split('\n')
gene_ids = set(gene_ids)

print(len(gene_ids), 'LRK10L gene IDs')

### Check LRK10Ls missing from records

In [ ]:
missing = gene_ids - set(rec.id for rec in records)
print(f'{len(missing)} LRK10L gene IDs not found in FASTA records')
if missing:
    print(list(missing)[:10])

## Length filtering

Sequences <250aa cannot be kinases, yet kinase filtering still misses some.

In [ ]:
MIN_SEQUENCE_LENGTH = 250
n_records_before = len(records)
records = [rec for rec in records if len(rec.seq) >= MIN_SEQUENCE_LENGTH]
record_species = {rec.id: record_species[rec.id] for rec in records}

print(f'Dropped {n_records_before - len(records)} sequences shorter than {MIN_SEQUENCE_LENGTH} aa')
print(f'{len(records)} FASTA records remaining')

## Kinase filtering

### Filter all sequences

In [ ]:
pkinase_hits = set()

files = os.listdir('../data/sequences')
for fname in tqdm(files, desc='Scanning proteomes'):
    filepath = Path('../data/sequences') / fname
    hits = search_proteome_for_kinases(filepath, hmms)
    print(f'{fname}: {len(hits)} kinase hits')
    pkinase_hits.update(hits)

print(f'{len(pkinase_hits)} total proteins with Pkinase hit across all species')

### Check kinase filtering results

In [ ]:
missing_pkinase = gene_ids - pkinase_hits
print(f'{len(missing_pkinase)} known LRK10L genes have NO Pkinase domain hit')
if missing_pkinase:
    print(list(missing_pkinase))

### Drop non-kinases from records

In [ ]:
records = [rec for rec in records if rec.id in pkinase_hits]
labels = {rec.id: (1 if rec.id in gene_ids else 0) for rec in records}
print(f'{len(records)} proteins retained after Pkinase filter')

## Leave-one-group-out

In [ ]:
logo = LeaveOneGroupOut()

## Dedup

Remove duplicates within each species using cd-hit

Aug 16: This doesn't help much.

In [ ]:
RUN_DEDUP = True  #@param {type:"boolean"}

### Cluster duplicates

In [ ]:
if RUN_DEDUP:
    def check_redundancy(species_name, records, record_species, labels, out_dir='../intermediate/cdhit', c=0.90, n=5):
        pos_records = [
            rec for rec in records
            if record_species[rec.id] == species_name and labels[rec.id] == 1
        ]
        if len(pos_records) < 2:
            return {'species': species_name, 'n_input': len(pos_records), 'n_clusters': None, 'redundancy': None}

        os.makedirs(out_dir, exist_ok=True)
        safe_name = species_name.replace(' ', '_')
        fasta_path = f'{out_dir}/{safe_name}_positives.fasta'
        out_path = f'{out_dir}/{safe_name}'

        SeqIO.write(pos_records, fasta_path, 'fasta')

        !cd-hit -i {fasta_path} -o {out_path} -c {c} -n {n} -d 0 -aL 0.8 -aS 0.8 > /dev/null

        n_clusters = sum(1 for _ in SeqIO.parse(out_path, 'fasta'))
        redundancy = 1 - n_clusters / len(pos_records)
        return {
            'species': species_name,
            'n_input': len(pos_records),
            'n_clusters': n_clusters,
            'redundancy': redundancy,
        }

    results = []
    for species_name in sorted(set(record_species.values())):
        res = check_redundancy(species_name, records, record_species, labels)
        results.append(res)
    redundancy_df = pd.DataFrame(results).sort_values('redundancy', ascending=False)
    display(redundancy_df)

### Parse cd-hit output

In [ ]:
if RUN_DEDUP:
    def clusters_to_dataframe(clstr_path, species_name):
        """Parse a .clstr file into a DataFrame."""
        clusters = []
        current = None
        with open(clstr_path) as f:
            for line in f:
                line = line.strip()
                if line.startswith('>Cluster'):
                    if current is not None:
                        clusters.append(current)
                    current = []
                else:
                    idx, rest = line.split('\t')
                    length = int(rest.split(',')[0].strip().rstrip('aa'))
                    seq_id = rest.split('>')[1].split('...')[0]
                    is_rep = '*' in rest
                    identity = None if is_rep else rest.split('at')[-1].strip()
                    current.append({
                        'seq_id': seq_id,
                        'length': length,
                        'is_representative': is_rep,
                        'identity_to_rep': identity,
                    })
            if current is not None:
                clusters.append(current)
        rows = []
        for i, cluster in enumerate(clusters):
            for m in cluster:
                rows.append({
                    'species': species_name,
                    'cluster_id': i,
                    'cluster_size': len(cluster),
                    **m,
                })
        return pd.DataFrame(rows)

    all_cluster_dfs = []
    for species_name in sorted(set(record_species.values())):
        safe_name = species_name.replace(' ', '_')
        clstr_path = f'../intermediate/cdhit/{safe_name}.clstr'
        if not os.path.exists(clstr_path):
            continue
        df = clusters_to_dataframe(clstr_path, species_name)
        all_cluster_dfs.append(df)
    all_clusters_df = pd.concat(all_cluster_dfs, ignore_index=True)
    redundant_df = all_clusters_df[all_clusters_df['cluster_size'] > 1].sort_values(
        ['species', 'cluster_size', 'cluster_id', 'is_representative'], ascending=[True, False, True, False]
    )

### Drop duplicates

In [ ]:
if RUN_DEDUP:
    ids_to_drop = set(
        all_clusters_df.loc[
            (~all_clusters_df['is_representative']) & (all_clusters_df['cluster_size'] > 1),
            'seq_id'
        ]
    )
    print(f'{len(ids_to_drop)} sequences to drop as redundant')

    records_deduped = [rec for rec in records if rec.id not in ids_to_drop]
    print(f'{len(records_deduped)} records remaining after dropping redundant sequences')
else:
    records_deduped = records
    print(f'{len(records_deduped)} records (dedup skipped)')

labels_deduped = {rec.id: (1 if rec.id in gene_ids else 0) for rec in records_deduped}
y_deduped = np.array([labels[rec.id] for rec in records_deduped])
groups_deduped = np.array([record_species[rec.id] for rec in records_deduped])

## HMM baseline

Three-way outer split by species: use N-2 species to build the HMM, one validation species to select the e-value threshold, and one test species for final metrics.

In [ ]:
LOAD_HMM_RESULTS = True  #@param {"type": "boolean"}

### Align, build, scan

In [ ]:
hmm_baseline_dir = '../intermediate/hmm_baseline'
hmm_results_name = '../saves/hmm_results_df.csv'
os.makedirs(hmm_baseline_dir, exist_ok=True)

In [ ]:
if not LOAD_HMM_RESULTS:
    hmm_df = pd.DataFrame({
        'record': records_deduped,
        'species': [record_species[rec.id] for rec in records_deduped],
        'label': [labels_deduped[rec.id] for rec in records_deduped],
    })
    hmm_results = []
    all_species = sorted(hmm_df['species'].unique())

    def get_validation_species(all_species, test_species):
        others = [species for species in all_species if species != test_species]
        index = all_species.index(test_species) % len(others)
        return others[index]

    pbar = tqdm(all_species, desc='Evaluate HMM')
    for test_species in pbar:
        validation_species = get_validation_species(all_species, test_species)
        train_species = [
            species for species in all_species
            if species not in {validation_species, test_species}
        ]

        train_df = hmm_df[hmm_df['species'].isin(train_species)]
        validation_df = hmm_df[hmm_df['species'] == validation_species]
        test_df = hmm_df[hmm_df['species'] == test_species]

        species_pretty = test_species.split('_')[0]
        validation_species_pretty = validation_species.split('_')[0]
        pbar.set_postfix_str(species_pretty)

        train_fa_name = f'{hmm_baseline_dir}/{species_pretty}_train.fasta'
        validation_fa_name = f'{hmm_baseline_dir}/{species_pretty}_validation.fasta'
        test_fa_name = f'{hmm_baseline_dir}/{species_pretty}_test.fasta'
        aligned_fa_name = f'{hmm_baseline_dir}/{species_pretty}_train_aligned.fasta'

        SeqIO.write(
            list(train_df[train_df['label'] == 1]['record']),
            train_fa_name,
            'fasta',
        )
        SeqIO.write(list(validation_df['record']), validation_fa_name, 'fasta')
        SeqIO.write(list(test_df['record']), test_fa_name, 'fasta')

        !muscle -align {train_fa_name} -output {aligned_fa_name} -quiet

        alphabet = pyhmmer.easel.Alphabet.amino()
        with pyhmmer.easel.MSAFile(aligned_fa_name, digital=True, alphabet=alphabet) as msa_file:
            msa = msa_file.read()
            msa.name = f'LRK10L_{species_pretty}'

            builder = pyhmmer.plan7.Builder(alphabet)
            background = pyhmmer.plan7.Background(alphabet)
            hmm, _, _ = builder.build_msa(msa, background)

        for split_name, split_species, split_fa_name, split_df in [
            ('validation', validation_species, validation_fa_name, validation_df),
            ('test', test_species, test_fa_name, test_df),
        ]:
            with pyhmmer.easel.SequenceFile(split_fa_name, digital=True, alphabet=alphabet) as seq_file:
                split_seqs = list(seq_file)

            for seq, hits in zip(split_seqs, pyhmmer.hmmscan(split_seqs, [hmm], E=1e-4)):
                best_hit = min(hits, key=lambda h: h.evalue, default=None)
                hmm_results.append({
                    'outer_test_species': test_species,
                    'validation_species': validation_species,
                    'species': split_species,
                    'split': split_name,
                    'record_id': seq.name,
                    'pred_label': 1 if best_hit is not None else 0,
                    'true_label': labels_deduped[seq.name],
                    'evalue': best_hit.evalue if best_hit is not None else None,
                })

    hmm_results_df = pd.DataFrame(hmm_results)
    hmm_results_df.to_csv(hmm_results_name, index=False)
else:
    hmm_results_df = pd.read_csv(hmm_results_name)

### Visualize log e-values by label

In [ ]:
hmm_results_df['log_evalue'] = np.log10(hmm_results_df['evalue'].replace(0, 1e-300))

sns.histplot(data=hmm_results_df, x='log_evalue', hue='true_label', bins=200, stat='density', common_norm=False)
hmm_results_df.groupby('true_label')['evalue'].describe()

### Generate metrics

In [ ]:
hmm_results_df['evalue_filled'] = hmm_results_df['evalue'].fillna(1e10)


def best_f1_evalue_threshold(y_true, evalues):
    candidates = np.unique(evalues)
    best_threshold, best_f1 = float(candidates[0]), -1.0
    for threshold in candidates:
        f1 = f1_score(y_true, evalues <= threshold, zero_division=0)
        if f1 > best_f1:
            best_threshold, best_f1 = float(threshold), float(f1)
    return best_threshold, best_f1


all_test_species = sorted(hmm_results_df['outer_test_species'].unique())
hmm_fold_rows = []
hmm_fold_preds = {}

for test_species in all_test_species:
    fold_rows = hmm_results_df[hmm_results_df['outer_test_species'] == test_species]
    validation_rows = fold_rows[fold_rows['split'] == 'validation']
    test_rows = fold_rows[fold_rows['split'] == 'test']

    validation_threshold, validation_f1 = best_f1_evalue_threshold(
        validation_rows['true_label'].to_numpy(),
        validation_rows['evalue_filled'].to_numpy(),
    )
    test_y = test_rows['true_label'].to_numpy()
    test_evalues = test_rows['evalue_filled'].to_numpy()
    test_pred = test_evalues <= validation_threshold

    hmm_results_df.loc[fold_rows.index, 'fold_threshold'] = validation_threshold
    hmm_results_df.loc[fold_rows.index, 'pred_label'] = (
        hmm_results_df.loc[fold_rows.index, 'evalue_filled'] <= validation_threshold
    ).astype(int)
    hmm_fold_preds[test_species] = {
        'y_true': test_y,
        'preds': -test_evalues,
    }

    hmm_fold_rows.append({
        'species': test_species.split('_')[0],
        'validation_species': validation_rows['species'].iloc[0].split('_')[0],
        'threshold': validation_threshold,
        'val_f1_at_threshold': validation_f1,
        'test_pr_auc': average_precision_score(test_y, -test_evalues),
        'test_precision': precision_score(test_y, test_pred, zero_division=0),
        'test_recall': recall_score(test_y, test_pred, zero_division=0),
        'test_accuracy': accuracy_score(test_y, test_pred),
        'n_pos': int(test_y.sum()),
        'found': int(((test_y == 1) & test_pred).sum()),
        'false_positives': int(((test_y == 0) & test_pred).sum()),
    })

hmm_fold_results = pd.DataFrame(hmm_fold_rows)

In [ ]:
hmm_fold_results = hmm_fold_results.assign(
    precision=hmm_fold_results['test_precision'],
    recall=hmm_fold_results['test_recall'],
    accuracy=hmm_fold_results['test_accuracy'],
)

hmm_fold_results[[
    'species', 'validation_species', 'threshold', 'val_f1_at_threshold',
    'test_pr_auc', 'test_precision', 'test_recall', 'test_accuracy',
    'n_pos', 'found', 'false_positives'
]]

## [ESM] Build dataset

In [ ]:
LOAD_ESM = True  #@param {type:"boolean"}

### Load ESM model

In [ ]:
esm_model, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
batch_converter = alphabet.get_batch_converter()
esm_model.eval()

device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
esm_model = esm_model.to(device)

### Generate or load embeddings

In [ ]:
def esm_embed_records(records, esm_model, alphabet, batch_converter, device,
                           batch_size=8, max_len=1022):
    esm_model.eval()
    embeddings = {}

    print(f"Using device: {device}")

    # ESM has a length limit; truncate very long sequences
    data = [(rec.id, str(rec.seq)[:max_len]) for rec in records]
    data = sorted(data, key=lambda x: len(x[1]))

    for i in tqdm(range(0, len(data), batch_size), desc="Embedding batches"):
        batch = data[i:i+batch_size]
        labels, strs, tokens = batch_converter(batch)
        tokens = tokens.to(device)

        with torch.no_grad():
            out = esm_model(tokens, repr_layers=[esm_model.num_layers])
            reps = out['representations'][esm_model.num_layers]  # (batch, seq_len, embed_dim)

        for j, (label, seq) in enumerate(batch):
            # mean-pool over real residues, skipping BOS/EOS/padding tokens
            seq_len = len(seq)
            emb = reps[j, 1:seq_len+1].mean(0)  # +1 to skip BOS token
            embeddings[label] = emb.cpu().numpy()

    return embeddings

ESM_MODEL_NAME = 'esm2_t30_150M_UR50D'
ESM_SAVE_PATH = Path(f'../saves/{ESM_MODEL_NAME}_embeddings.npz')

def save_embeddings(embeddings: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    ids = np.array(list(embeddings.keys()))
    vecs = np.stack([embeddings[i] for i in ids])
    np.savez_compressed(path, ids=ids, embeddings=vecs)
    print(f'Saved {len(ids)} embeddings to {path}')

def load_embeddings(path: Path) -> dict:
    data = np.load(path, allow_pickle=False)
    ids, vecs = data['ids'], data['embeddings']
    return {i: v for i, v in zip(ids, vecs)}

all_ids = set(rec.id for rec in records)

if LOAD_ESM and ESM_SAVE_PATH.exists():
    embeddings_full_esm = load_embeddings(ESM_SAVE_PATH)
    missing_ids = all_ids - set(embeddings_full_esm.keys())

    if missing_ids:
        print(f'{len(missing_ids)} records missing from saved embeddings, generating those')
        missing_records = [rec for rec in records if rec.id in missing_ids]
        new_embeddings = esm_embed_records(missing_records, esm_model, alphabet, batch_converter, device)
        embeddings_full_esm.update(new_embeddings)
        save_embeddings(embeddings_full_esm, ESM_SAVE_PATH)  # persist the merged set
    else:
        print(f'Loaded {len(embeddings_full_esm)} embeddings from {ESM_SAVE_PATH}, all records covered')
else:
    embeddings_full_esm = esm_embed_records(records, esm_model, alphabet, batch_converter, device)
    save_embeddings(embeddings_full_esm, ESM_SAVE_PATH)

X_deduped_esm = pd.DataFrame(
    [embeddings_full_esm[rec.id] for rec in records_deduped],
    index=[rec.id for rec in records_deduped]
)
print(X_deduped_esm.shape)

### Keep embeddings for full data

In [ ]:
# Save copy of full ESM feature matrix
X_esm = pd.DataFrame(
    [embeddings_full_esm[rec.id] for rec in records],
    index=[rec.id for rec in records]
)

y = np.array([labels[rec.id] for rec in records])
groups = np.array([record_species[rec.id] for rec in records])

print(X_esm.shape, 'full post-kinase-filter pre-dedup ESM dataset')

## [k-mer] Build dataset

### Build k-mer vectors

In [ ]:
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
K = 2

all_kmers = [''.join(p) for p in itertools.product(AMINO_ACIDS, repeat=K)]
kmer_index = {kmer: i for i, kmer in enumerate(all_kmers)}
print(f'{len(all_kmers)} possible {K}-mers')

def kmer_freq_vector(seq, k, kmer_index):
    seq = str(seq).upper()
    vec = np.zeros(len(kmer_index))
    count = 0
    for i in range(len(seq) - k + 1):
        kmer = seq[i:i+k]
        if kmer in kmer_index:  # skip ambiguous
            vec[kmer_index[kmer]] += 1
            count += 1
    if count > 0:
        vec /= count  # normalize
    return vec

X_deduped_kmer = pd.DataFrame(
    [kmer_freq_vector(rec.seq, K, kmer_index) for rec in records_deduped],
    columns=all_kmers,
    index=[rec.id for rec in records_deduped]
)
print(X_deduped_kmer.shape)
display(X_deduped_kmer.head())

### Build k-mer vectors for full data

This is needed for some analysis requiring un-deduped data.

In [ ]:
X_kmer = pd.DataFrame(
    [kmer_freq_vector(rec.seq, K, kmer_index) for rec in records],
    columns=all_kmers,
    index=[rec.id for rec in records]
)
y = np.array([labels[rec.id] for rec in records])
groups = np.array([record_species[rec.id] for rec in records])

print(X_kmer.shape, 'full post-kinase-filter pre-dedup dataset')

## Check dataset

In [ ]:
zero_rows = (X_deduped_kmer.sum(axis=1) == 0).sum()
print(f'{zero_rows} sequences with no valid k-mers extracted')

print(len(groups), 'group labels built')
print(len(set(groups)), 'unique species in groups')

pos_species = pd.Series([record_species[rec.id] for rec in records_deduped if labels_deduped[rec.id] == 1])
print(pos_species.value_counts())

## Positive-unlabeled bagging

https://academic.oup.com/bib/article/23/1/bbab461/6415313

Should be ran per LOGO fold to avoid data leak but it takes too long, could do a one-fold test occcasionally instead.

In [ ]:
RUN_PU = True  #@param {type: "boolean"}

### PU global diagnostic

In [ ]:
if RUN_PU:
    N_ITER = 100
    NEG_SAMPLE_RATIO = 3

    pos_idx_all = np.where(y_deduped == 1)[0]
    neg_idx_all = np.where(y_deduped == 0)[0]

    pu_scores_diag, n_oob_diag = run_pu_bagging(
        X_deduped_kmer, y_deduped, pos_idx_all, neg_idx_all,
        n_iter=N_ITER, neg_sample_ratio=NEG_SAMPLE_RATIO, seed=2026,
    )

    pu_df_diagnostic = pd.DataFrame({
        'id': X_deduped_kmer.index[neg_idx_all],
        'species': groups_deduped[neg_idx_all],
        'pu_score': pu_scores_diag,
        'n_oob': n_oob_diag,
    }).sort_values('pu_score', ascending=False)
    display(pu_df_diagnostic.groupby('species').head())

### PU score distribution

In [ ]:
if RUN_PU:
    plt.figure(figsize=(8, 4))
    plt.hist(pu_df_diagnostic['pu_score'], bins=100)
    plt.xlabel('Average PU score (bagged)')
    plt.ylabel('Count')
    plt.title('Distribution of PU scores among labeled-negative proteins')
    plt.show()

    display(pu_df_diagnostic.groupby('species')['pu_score'].describe())

### Do HMM FPs match PU-flagged candidates?

Ensure this is updated to match HMM results.

In [ ]:
if RUN_PU:
    pu_candidates = set(pu_df_diagnostic.loc[pu_df_diagnostic['pu_score'] >= 0.80, 'id'])

    hmm_false_positive_ids = set("""
LOC_Os01g26210.1
LOC_Os02g42110.1
LOC_Os02g42160.1
LOC_Os02g42190.1
LOC_Os04g03830.1
LOC_Os04g29580.1
LOC_Os10g09620.1
VIT_200s0294g00020.1
VIT_200s0294g00080.1
VIT_200s0424g00020.1
VIT_201s0011g03960.1
VIT_201s0011g03980.1
VIT_201s0026g01420.1
VIT_208s0007g00960.1
VIT_208s0007g04350.1
VIT_213s0019g02300.1
VIT_214s0006g02600.2
VIT_216s0013g01198.1
VIT_216s0013g01235.1
VIT_216s0013g01386.1
VIT_217s0000g03340.1
Solyc01T000275.1
Solyc05T000170.1
Solyc11T000232.1
Solyc11T000233.1
Glyma04g13054.1
Glyma18g53186.1
mrna02191.1-v1.0-hybrid
mrna11412.1-v1.0-hybrid
Phvul.001G076300.1.p
Phvul.007G031300.1.p
    """.strip().split('\n'))
    overlap = pu_candidates & hmm_false_positive_ids
    print(f'{len(overlap)} of {len(pu_candidates)} PU candidates overlap with HMM false positives')
    print(overlap)

## [XGB] Model

### Positive scale weight

In [ ]:
n_pos = (y_deduped == 1).sum()
n_neg = (y_deduped == 0).sum()
print(f'{n_pos} positive, {n_neg} negative')
print(f"Positive rate: {n_pos / len(labels_deduped):.3%}")

### XGBoost model

In [ ]:
def make_model(scale_pos_weight, random_state=2026):
    return XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric=['aucpr'],

        n_estimators=1200,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=6,
        gamma=0,

        random_state=random_state,
        n_jobs=-1,
    )

class XGBEvaluator(PUFilterMixin, LOGOEvaluator):
    def make_model_per_split(self, X_train, y_train):
        scale_pos_weight = get_scale_pos_weight(y_train)
        return make_model(scale_pos_weight)

    def fit_model_per_split(self, model, X_train, y_train, X_val, y_val):
        model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            verbose=False,
        )
        return model

    def extra_metrics_per_split(self, model, X_train, y_train, X_val, y_val):
        evals = model.evals_result()
        val_curve = evals['validation_1']['aucpr']
        best_round = int(np.argmax(val_curve))
        return {
            'best_round': best_round,
            'best_val_pr_auc': val_curve[best_round],
            'final_val_pr_auc': val_curve[-1],
            'n_flagged': getattr(self, '_last_n_flagged', None),
        }

    def extra_artifacts_per_split(self, model, X_train, y_train, X_val, y_val,
                                   X_val_full, y_val_full, preds_full):
        artifacts = {
            'evals': model.evals_result(),
            'importance': pd.Series(
                model.feature_importances_, index=X_train.columns
            ).sort_values(ascending=False),
        }
        return artifacts

### Fit

In [ ]:
# Fit XGBoost with k-mer
xgb_evaluator = XGBEvaluator(
    logo=logo,
    X=X_deduped_kmer,
    y=y_deduped,
    groups=groups_deduped,
    auto_threshold=True,
    X_full=X_kmer,
    y_full=y,
    groups_full=groups,
    run_pu=RUN_PU
)
xgb_evaluator.fit()

xgb_evaluator.results_df[[
    'species', 'validation_species', 'threshold', 'val_f1_at_threshold',
    'test_pr_auc', 'test_precision', 'test_recall', 'n_pos', 'found',
    'false_positives'
]]

In [ ]:
test_precision = xgb_evaluator.results_df['test_precision']
test_recall = xgb_evaluator.results_df['test_recall']

print("XGB model (held-out test species)")
print(f"Test precision: {test_precision.mean():.3f}±{test_precision.std():.3f}")
print(f"Test recall:    {test_recall.mean():.3f}±{test_recall.std():.3f}")

### Learning curves

In [ ]:
species_list = list(xgb_evaluator.fold_artifacts.keys())
n = len(species_list)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)

for i, species in enumerate(species_list):
    ax = axes[i // ncols][i % ncols]
    evals = xgb_evaluator.fold_artifacts[species]['evals']
    train_curve = evals['validation_0']['aucpr']
    validation_curve = evals['validation_1']['aucpr']
    rounds = range(1, len(train_curve) + 1)

    ax.plot(rounds, train_curve, label='train', color='tab:blue')
    ax.plot(rounds, validation_curve, label='validation', color='tab:orange')
    best_round = int(xgb_evaluator.fold_results[species]['best_round']) + 1
    ax.axvline(best_round, color='gray', linestyle='--', linewidth=1, label=f'best round ({best_round})')
    ax.set_title(f'{species.split("_")[0]} (test species)')
    ax.set_xlabel('boosting round')
    ax.set_ylabel('PR-AUC')
    ax.legend(fontsize=8)

for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis('off')

plt.tight_layout()
plt.show()

### Precision-recall graphs

In [ ]:
species_list = list(xgb_evaluator.fold_preds.keys())
n = len(species_list)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)

for i, species in enumerate(species_list):
    ax = axes[i // ncols][i % ncols]
    data = xgb_evaluator.fold_preds[species]
    precisions, recalls, thresholds = precision_recall_curve(data['y_true'], data['preds'])

    ax.plot(recalls, precisions, color='tab:blue')
    ax.set_title(f'{species.split("_")[0]} (held out)')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')

for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis('off')

plt.tight_layout()
plt.show()

### Precision-recall aggregated

In [ ]:
all_y_true = np.concatenate([xgb_evaluator.fold_preds[s]['y_true'] for s in xgb_evaluator.fold_preds])
all_preds = np.concatenate([xgb_evaluator.fold_preds[s]['preds'] for s in xgb_evaluator.fold_preds])

precisions_xgb, recalls_xgb, thresholds_xgb = precision_recall_curve(all_y_true, all_preds)
pr_auc = average_precision_score(all_y_true, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))

for species, data in xgb_evaluator.fold_preds.items():
    p, r, _ = precision_recall_curve(data['y_true'], data['preds'])
    ax.plot(r, p, color='gray', alpha=0.2, linewidth=1)

ax.plot(recalls_xgb, precisions_xgb, color='tab:blue', linewidth=2.5, label=f'pooled AP={pr_auc:.3f}')

ax.set_title('Pooled PR curve for all held-out species')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Recall at FP budgets

In [ ]:
def recall_at_fp_budgets(y_true, preds, fp_budgets):
    order = np.argsort(-preds)
    y_sorted = y_true[order]

    n_pos = y_sorted.sum()
    cum_tp = np.cumsum(y_sorted == 1)
    cum_fp = np.cumsum(y_sorted == 0)

    results = {}
    for budget in fp_budgets:
        eligible = np.where(cum_fp <= budget)[0]
        if len(eligible) == 0:
            results[budget] = {'found': 0, 'recall': 0.0, 'actual_fp': 0}
            continue
        idx = eligible[-1]
        results[budget] = {
            'found': int(cum_tp[idx]),
            'recall': cum_tp[idx] / n_pos,
            'actual_fp': int(cum_fp[idx]),
        }
    return results

fp_budgets = [5, 10, 15]

recall_rows = {}
for species, data in xgb_evaluator.fold_preds.items():
    per_budget = recall_at_fp_budgets(data['y_true'], data['preds'], fp_budgets)
    for budget, stats in per_budget.items():
        recall_rows[(species.split('_')[0], budget)] = stats

recall_df = pd.DataFrame(recall_rows).T
recall_df.index.names = ['species', 'fp_budget']
recall_df

### Species-specific split details

In [ ]:
def specific_results(species_full: str):
    species_short = species_full.split('_')[0]
    df = xgb_evaluator.fold_details[species_full]
    threshold = xgb_evaluator.fold_results[species_full]['threshold']

    predicted_positive = df['pred'] >= threshold
    hits = df[(df['y_true'] == 1) & predicted_positive]
    misses = df[(df['y_true'] == 1) & ~predicted_positive]
    false_positives = df[(df['y_true'] == 0) & predicted_positive]

    print(f"\n{'='*10} {species_short} (test species; threshold={threshold:.3f}) {'='*10}")
    print(f'Hits ({len(hits)}):\n{hits}')
    print(f'\nMisses ({len(misses)}):\n{misses}')
    print(f'\nFalse positives ({len(false_positives)}):\n{false_positives}')

specific_results('Slycopersicum_796_ITAG5.0.protein_primaryTranscriptOnly')
specific_results('Athaliana_447_Araport11.protein_primaryTranscriptOnly')
specific_results('Osativa_204_v7.0.protein_primaryTranscriptOnly')
specific_results('Vvinifera_457_v2.1.protein_primaryTranscriptOnly')
specific_results('Gmax_189_protein_primaryTranscriptOnly')
specific_results('Fvesca_226_v1.1.protein_primaryTranscriptOnly')
specific_results('Pvulgaris_442_v2.1.protein_primaryTranscriptOnly')

### Feature importance per split

#### Importance Spearman correlation

In [ ]:
importance_df = pd.DataFrame({
    species.split('_')[0]: artifacts['importance']
    for species, artifacts in xgb_evaluator.fold_artifacts.items()
})

corr_matrix = importance_df.corr(method="spearman")
plt.figure(figsize=(6, 3))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.show()

#### Top features Jaccard similarity

In [ ]:
def top_k_jaccard(importance_df, k=20):
    top_sets = {
        species: set(importance_df[species].sort_values(ascending=False).head(k).index)
        for species in importance_df.columns
    }

    species_list = list(top_sets.keys())
    results = {}
    for i in range(len(species_list)):
        for j in range(i+1, len(species_list)):
            a, b = species_list[i], species_list[j]
            intersection = top_sets[a] & top_sets[b]
            union = top_sets[a] | top_sets[b]
            results[(a, b)] = len(intersection) / len(union)

    return results, top_sets

jaccard_scores, top_sets = top_k_jaccard(importance_df, k=20)
for pair, score in jaccard_scores.items():
    print(f"{pair[0]} vs {pair[1]}: {score:.2f}")

common_across_all = set.intersection(*top_sets.values())
print(f"\n{len(common_across_all)} features in every fold's top-20:")
print(common_across_all)

## [LogReg] Model

A simpler model than XGBoost, same purpose as cosine similarity above.

### Train model

In [ ]:
class LogRegEvaluator(PUFilterMixin, LOGOEvaluator):
    def make_model_per_split(self, X_train, y_train):
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight='balanced', max_iter=1000, random_state=2026, C=0.1)
        )

    def extra_metrics_per_split(self, model, X_train, y_train, X_val, y_val):
        return {'n_flagged': getattr(self, '_last_n_flagged', None)}

# Build logistic regression with ESM-2
logreg_evaluator = LogRegEvaluator(
    logo=logo,
    X=X_deduped_esm,
    y=y_deduped,
    groups=groups_deduped,
    auto_threshold=True,
    X_full=X_esm,
    y_full=y,
    groups_full=groups,
    run_pu=RUN_PU,
).fit()

logreg_evaluator.results_df[[
    'species', 'validation_species', 'threshold', 'val_f1_at_threshold',
    'test_pr_auc', 'test_precision', 'test_recall', 'n_pos', 'found',
    'false_positives'
]]

In [ ]:
test_precision = logreg_evaluator.results_df['test_precision']
test_recall = logreg_evaluator.results_df['test_recall']

print("LogReg model (held-out test species)")
print(f"Test precision: {test_precision.mean():.3f}±{test_precision.std():.3f}")
print(f"Test recall:    {test_recall.mean():.3f}±{test_recall.std():.3f}")

### Precision-recall aggregated

In [ ]:
logreg_fold_preds = logreg_evaluator.fold_preds

all_y_true_logreg = np.concatenate([logreg_fold_preds[s]['y_true'] for s in logreg_fold_preds])
all_preds_logreg = np.concatenate([logreg_fold_preds[s]['preds'] for s in logreg_fold_preds])

precisions_logreg, recalls_logreg, thresholds_logreg = precision_recall_curve(all_y_true_logreg, all_preds_logreg)
pr_auc_logreg = average_precision_score(all_y_true_logreg, all_preds_logreg)

fig, ax = plt.subplots(figsize=(6, 5))

for species, data in logreg_fold_preds.items():
    p, r, _ = precision_recall_curve(data['y_true'], data['preds'])
    ax.plot(r, p, color='gray', alpha=0.2, linewidth=1)

ax.plot(
    recalls_logreg, precisions_logreg, color='tab:blue', linewidth=2.5,
    label=f'pooled AP={pr_auc_logreg:.3f}',
)

ax.set_title('Pooled PR curve for LogReg across held-out species')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Meta-model

Combine k-mer XGB and ESM LogReg

### Make meta dataset

In [ ]:
xgb_all = pd.concat(xgb_evaluator.fold_details.values()).set_index('id')
lr_all = pd.concat(logreg_evaluator.fold_details.values()).set_index('id')

meta_df = pd.DataFrame({
    'xgb_pred': xgb_all['pred'],
    'lr_pred': lr_all['pred'],
    'y_true': xgb_all['y_true'],
}).dropna()

assert (meta_df['y_true'] == lr_all.loc[meta_df.index, 'y_true']).all(), 'label mismatch between models'

meta_df['diff'] = meta_df['xgb_pred'] - meta_df['lr_pred']
meta_df['abs_diff'] = meta_df['diff'].abs()
meta_df['mean_pred'] = (meta_df['xgb_pred'] + meta_df['lr_pred']) / 2

meta_df['xgb_logit'] = np.log(meta_df['xgb_pred'] / (1 - meta_df['xgb_pred']))
meta_df['lr_logit'] = np.log(meta_df['lr_pred'] / (1 - meta_df['lr_pred']))

species_lookup = pd.Series(groups, index=X_kmer.index, name='species')
meta_df['species'] = species_lookup.reindex(meta_df.index)

assert meta_df['species'].notna().all(), "some ids couldn't be matched to a species"

X_meta = meta_df[['xgb_logit', 'lr_logit']]
y_meta = meta_df['y_true'].values
groups_meta = meta_df['species'].values

X_meta.head()

### Fit

In [ ]:
class MetaEvaluator(LOGOEvaluator):
    def make_model_per_split(self, X_train, y_train):
        return LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=2026,
            C=0.1,
        )

meta_evaluator = MetaEvaluator(
    logo=logo,
    X=X_meta,
    y=y_meta,
    groups=groups_meta,
    auto_threshold=True,
)
meta_evaluator.fit()
meta_evaluator.results_df[[
    'species', 'validation_species', 'threshold', 'val_f1_at_threshold',
    'test_pr_auc', 'test_precision', 'test_recall', 'n_pos', 'found',
    'false_positives'
]]

### Pooled PR curve

In [ ]:
meta_fold_preds = meta_evaluator.fold_preds

all_y_true_meta = np.concatenate([meta_fold_preds[s]['y_true'] for s in meta_fold_preds])
all_preds_meta = np.concatenate([meta_fold_preds[s]['preds'] for s in meta_fold_preds])

precisions_meta, recalls_meta, thresholds_meta = precision_recall_curve(all_y_true_meta, all_preds_meta)
pr_auc_meta = average_precision_score(all_y_true_meta, all_preds_meta)

fig, ax = plt.subplots(figsize=(6, 5))

for species, data in meta_fold_preds.items():
    p, r, _ = precision_recall_curve(data['y_true'], data['preds'])
    ax.plot(r, p, color='gray', alpha=0.2, linewidth=1)

ax.plot(
    recalls_meta, precisions_meta, color='tab:blue', linewidth=2.5,
    label=f'pooled AP={pr_auc_meta:.3f}',
)

ax.set_title('Pooled PR curve for meta-model across held-out species')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Model comaprison

### Boxplots

In [ ]:
models = {
    'k-mer XGB': xgb_evaluator,
    'ESM LogReg': logreg_evaluator,
    'Meta': meta_evaluator,
}
model_labels = list(models)

precision = np.array([
    [result['test_precision'] for result in evaluator.fold_results.values()]
    for evaluator in models.values()
] + [hmm_fold_results['precision']])
recall = np.array([
    [result['test_recall'] for result in evaluator.fold_results.values()]
    for evaluator in models.values()
] + [hmm_fold_results['recall']])
accuracy = np.array([
    [result.get(
        'test_accuracy',
        accuracy_score(
            evaluator.fold_preds[species]['y_true'],
            evaluator.fold_preds[species]['preds'] >= result.get('threshold', 0.5),
        ),
    ) for species, result in evaluator.fold_results.items()]
    for evaluator in models.values()
] + [hmm_fold_results['accuracy']])
pr_auc = np.array([
    [result['test_pr_auc'] for result in evaluator.fold_results.values()]
    for evaluator in models.values()
] + [hmm_fold_results['test_pr_auc']])

metrics = {
    'PR-AUC': (pr_auc, model_labels + ['HMM']),
    'Precision': (precision, model_labels + ['HMM']),
    'Recall': (recall, model_labels + ['HMM']),
    'Accuracy': (accuracy, model_labels + ['HMM']),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (metric_name, (scores, labels)) in zip(axes.flat, metrics.items()):
    ax.boxplot(
        scores.T,
        tick_labels=labels
    )
    ax.set_title(metric_name)
    ax.set_ylabel('Score')
    # if metric_name != 'Accuracy':
    #     ax.set_ylim(0, 1)
    # else:
    #     ax.set_ylim(0.9, 1)

plt.tight_layout()
plt.show()

### Friedman & Nemenyi

https://www.jmlr.org/papers/volume7/demsar06a/demsar06a.pdf

In [ ]:
ALPHA = 0.05
METHODS = list(models.keys()) + ['HMM']

def _species_key(name):
    return str(name).split('_')[0]

def _test_metric(evaluator, species, result, metric):
    if metric in result:
        return result[metric]
    values = evaluator.fold_preds[species]
    y_true = values['y_true']
    preds = values['preds']
    hard = preds >= result.get('threshold', 0.5)
    if metric == 'test_pr_auc':
        return average_precision_score(y_true, preds)
    if metric == 'test_precision':
        return precision_score(y_true, hard, zero_division=0)
    if metric == 'test_recall':
        return recall_score(y_true, hard, zero_division=0)
    if metric == 'test_accuracy':
        return accuracy_score(y_true, hard)
    raise KeyError(metric)

fold_species = [_species_key(name) for name in models[METHODS[0]].fold_results.keys()]
hmm_species = hmm_fold_results['species'].astype(str).tolist()
if set(fold_species) != set(hmm_species):
    raise ValueError(f'Fold mismatch between ML models and HMM baseline: {fold_species} vs {hmm_species}')

metric_columns = {
    'PR-AUC': ('test_pr_auc', 'test_pr_auc'),
    'Accuracy': ('test_accuracy', 'accuracy'),
    'Precision': ('test_precision', 'precision'),
    'Recall': ('test_recall', 'recall'),
}
scores_by_metric = {}
for metric_name, (result_key, hmm_key) in metric_columns.items():
    score_frame = pd.DataFrame({
        method: {
            _species_key(species): float(_test_metric(evaluator, species, result, result_key))
            for species, result in evaluator.fold_results.items()
        }
        for method, evaluator in models.items()
    })
    hmm_values = hmm_fold_results.set_index('species')[hmm_key].astype(float)
    score_frame['HMM'] = hmm_values.reindex(score_frame.index)
    score_frame = score_frame.reindex(columns=METHODS)
    if score_frame.isna().any().any():
        raise ValueError(f'Missing {metric_name} scores after fold alignment')
    scores_by_metric[metric_name] = score_frame

friedman_summary = []
nemenyi_results = {}
cd_values = {}

fig, axes = plt.subplots(len(scores_by_metric), 1, figsize=(10, 3.2 * len(scores_by_metric)))
if len(scores_by_metric) == 1:
    axes = [axes]

for ax, (metric_name, score_frame) in zip(axes, scores_by_metric.items()):
    statistic, p_value = friedmanchisquare(*[score_frame[method] for method in METHODS])
    ranks = score_frame.rank(axis=1, ascending=False, method='average')
    rank_means = ranks.mean(axis=0)
    nemenyi_pvalues = sp.posthoc_nemenyi_friedman(-score_frame)
    nemenyi_pvalues = nemenyi_pvalues.reindex(index=METHODS, columns=METHODS)
    n_blocks, n_methods = ranks.shape
    cd = studentized_range.ppf(1 - ALPHA, n_methods, np.inf) / np.sqrt(2) * np.sqrt(
        n_methods * (n_methods + 1) / (6 * n_blocks)
    )
    friedman_summary.append({
        'metric': metric_name,
        'n_folds': n_blocks,
        'chi2': statistic,
        'p_value': p_value,
        'significant_at_0.05': p_value < ALPHA,
        'best_average_rank': rank_means.idxmin(),
    })
    cd_values[metric_name] = cd
    nemenyi_results[metric_name] = nemenyi_pvalues if p_value < ALPHA else None
    if p_value < ALPHA:
        print(f'{metric_name}: Friedman chi2={statistic:.3f}, p={p_value:.4g}')
    else:
        print(f'{metric_name}: Friedman chi2={statistic:.3f}, p={p_value:.4g}')
        print('  Omnibus test is not significant.')

    sp.critical_difference_diagram(
        rank_means,
        nemenyi_pvalues,
        cd=cd,
        alpha=ALPHA,
        ax=ax,
        label_props={'fontsize': 10},
    )
    ax.set_title(metric_name)

plt.tight_layout()
plt.show()

friedman_summary_df = pd.DataFrame(friedman_summary).set_index('metric')
display(friedman_summary_df.round(4))

## Failure analysis

In [ ]:
def get_miss_fp_sets(evaluator):
    misses, fps = set(), set()
    for species, details in evaluator.fold_details.items():
        threshold = evaluator.fold_results[species]['threshold']
        pred_pos = details['pred'] >= threshold
        misses.update(details[(details['y_true'] == 1) & ~pred_pos]['id'])
        fps.update(details[(details['y_true'] == 0) & pred_pos]['id'])
    return misses, fps

all_evaluators = {
    'lr_esm': logreg_evaluator,
    'xgb_kmer': xgb_evaluator,
    'meta': meta_evaluator,
}

miss_sets = {}
fp_sets = {}
for name, ev in all_evaluators.items():
    m, f = get_miss_fp_sets(ev)
    miss_sets[name] = m
    fp_sets[name] = f

miss_counter = Counter()
for s in miss_sets.values():
    miss_counter.update(s)

fp_counter = Counter()
for s in fp_sets.values():
    fp_counter.update(s)

n_models = len(all_evaluators)
print(f"Misses by agreement level (out of {n_models} models):")
print(pd.Series(miss_counter.values()).value_counts().sort_index())
print(f"\nFPs by agreement level (out of {n_models} models):")
print(pd.Series(fp_counter.values()).value_counts().sort_index())

# majority (2 of 3) as "hard" cases
majority_misses = {k for k,v in miss_counter.items() if v >= 2}
majority_fps = {k for k,v in fp_counter.items() if v >= 2}
print(f"\nMajority (2+/3) misses: {len(majority_misses)}")
print(f"Majority (2+/3) FPs: {len(majority_fps)}")

xgb_pooled_probs = pd.concat(xgb_evaluator.fold_details.values()).set_index('id')
uncertain = xgb_pooled_probs[(xgb_pooled_probs['pred'] > 0.35) & (xgb_pooled_probs['pred'] < 0.65)]
print(f"{len(uncertain)} examples XGB is uncertain about")

In [ ]:
species_lookup = pd.Series(groups, index=X_kmer.index)

miss_species = species_lookup.reindex(list(majority_misses)).value_counts()
fp_species = species_lookup.reindex(list(majority_fps)).value_counts()

n_pos_by_species = pd.Series(y, index=X_kmer.index)[pd.Series(y, index=X_kmer.index) == 1] \
    .groupby(species_lookup).size()

error_species_df = pd.DataFrame({
    'misses': miss_species,
    'fps': fp_species,
    'n_pos': n_pos_by_species,
}).fillna(0).astype(int)

error_species_df['total_errors'] = error_species_df['misses'] + error_species_df['fps']
error_species_df['error_rate'] = error_species_df['total_errors'] / error_species_df['n_pos']

error_species_df = error_species_df[['total_errors', 'misses', 'fps', 'n_pos', 'error_rate']] \
    .sort_values('error_rate', ascending=False)

print(error_species_df)

### Inspect tomato

In [ ]:
sly_mask = species_lookup.str.startswith('Slycopersicum')
sly_errors = (majority_misses | majority_fps) & set(species_lookup[sly_mask].index)
for e in sly_errors:
    print(e)

## --------------------------------------

## PU-clean full dataset

In [ ]:
if RUN_PU:
    pos_idx_full = np.where(y_deduped == 1)[0]
    neg_idx_full = np.where(y_deduped == 0)[0]

    pu_scores_full, n_oob_full = run_pu_bagging(
        X_deduped_kmer, y_deduped, pos_idx_full, neg_idx_full,
        n_iter=N_ITER, neg_sample_ratio=NEG_SAMPLE_RATIO, seed=2026,
    )
    flagged_full_mask = pu_scores_full >= 0.8
    flagged_full_ids = set(X_deduped_kmer.index[neg_idx_full[flagged_full_mask]])

    print(f'{len(flagged_full_ids)} proteins flagged for removal (pu_score >= 0.8)')

    clean_mask_full = ~X_deduped_kmer.index.isin(flagged_full_ids)
    X_clean_full = X_deduped_kmer[clean_mask_full]
    y_clean_full = y_deduped[clean_mask_full]
    groups_clean_full = groups_deduped[clean_mask_full]

    print(f'{len(y_deduped) - len(y_clean_full)} proteins removed')
    print(f'{y_clean_full.sum()} positives remain, {(y_clean_full == 0).sum()} negatives remain')
else:
    print('Did not run PU')
    X_clean_full = X_deduped_kmer.copy()
    y_clean_full = y_deduped.copy()
    groups_clean_full = groups_deduped.copy()

## Full model

Build a full model using all species and analyze features. Could also be used for extra test inference in the future.

### Fit full model

In [ ]:
scale_pos_weight_full = get_scale_pos_weight(y_clean_full)

model_full = make_model(scale_pos_weight_full)
model_full.fit(X_clean_full, y_clean_full)

print(f'{(y_clean_full == 1).sum()} positives, {(y_clean_full == 0).sum()} negatives')
print(f'scale_pos_weight_full = {scale_pos_weight_full:.2f}')

### Feature importance

In [ ]:
importances = model_full.feature_importances_
imp_df = pd.DataFrame({
    'kmer': X_clean_full.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

display(imp_df.head())

## Misc analysis

### Model quality / complexity

#### Baseline cosine

Establish a baseline using cosine similarity instead of a model.

"Does XGBoost actually need the complexity?"

In [ ]:
cos_fold_results = {}

for train_idx, val_idx in logo.split(X_deduped_kmer, y_deduped, groups=groups_deduped):
    held_out_species = groups_deduped[val_idx][0]
    n_pos_val = y_deduped[val_idx].sum()
    n_total_val = len(val_idx)
    baseline = n_pos_val / n_total_val

    X_train = X_deduped_kmer.iloc[train_idx]
    X_val = X_deduped_kmer.iloc[val_idx]

    centroid = X_train[y_deduped[train_idx] == 1].mean(axis=0)
    similarity_scores = cosine_similarity(X_val, centroid.values.reshape(1, -1)).flatten()
    score = average_precision_score(y_deduped[val_idx], similarity_scores)
    lift = score / baseline

    cos_fold_results[held_out_species] = {
        'baseline': baseline,
        'pr_auc': score,
        'lift': lift,
        'n_pos': n_pos_val,
    }

cos_results_df = pd.DataFrame(cos_fold_results).T
cos_results_df

#### Logistic regression for species identity

Check for potential phylogenetic leak rather than data leak.

"Is species identity easily identifiable from k-mer features?"

In [ ]:
species_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        random_state=2026,
    )
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
scores = cross_val_score(species_clf, X_deduped_kmer, groups_deduped, cv=cv, scoring='accuracy')

print(f'Species prediction accuracy (5-fold CV): {scores.mean():.3f} +/- {scores.std():.3f}')
print(f'Chance baseline (majority class): {pd.Series(groups_deduped).value_counts(normalize=True).max():.3f}')
print(f'Chance baseline (random guess, {len(set(groups_deduped))} classes): {1/len(set(groups_deduped)):.3f}')

### K-mer matrix similarity

#### Inter-species

In [ ]:
pos_mask = y_deduped == 1
X_pos = X_deduped_kmer[pos_mask]
species_pos = groups_deduped[pos_mask]
ids_pos = X_pos.index

sim_matrix = cosine_similarity(X_pos)

rows = []
for i in range(len(ids_pos)):
    for j in range(i+1, len(ids_pos)):
        if species_pos[i] != species_pos[j]:
            rows.append({
                'protein_a': ids_pos[i],
                'protein_b': ids_pos[j],
                'similarity': sim_matrix[i, j],
            })

sim_df = pd.DataFrame(rows).sort_values('similarity', ascending=False)
sim_df.head()

#### Intra-species, positive

In [ ]:
within_species_pos_rows = []

for species in sorted(set(groups_deduped)):
    mask = (groups_deduped == species) & (y_deduped == 1)
    X_sub = X_deduped_kmer[mask]
    ids_sub = X_sub.index
    if len(ids_sub) < 2:
        continue

    sim = cosine_similarity(X_sub)
    for i in range(len(ids_sub)):
        for j in range(i + 1, len(ids_sub)):
            within_species_pos_rows.append({
                'species': species.split('_')[0],
                'protein_a': ids_sub[i],
                'protein_b': ids_sub[j],
                'similarity': sim[i, j],
            })

within_species_pos_sim = pd.DataFrame(within_species_pos_rows)
top_intra_sim = within_species_pos_sim[within_species_pos_sim['similarity'] >= 0.95].sort_values('similarity', ascending=False)
print(len(top_intra_sim))
top_intra_sim

#### Intra-species, negative, random sample

In [ ]:
rng = np.random.default_rng(2026)

def sampled_pairwise_sim(X_sub, ids_sub, n_pairs=2000):
    n = len(ids_sub)
    vecs = X_sub.values
    idx_a = rng.integers(0, n, size=n_pairs)
    idx_b = rng.integers(0, n, size=n_pairs)
    keep = idx_a != idx_b
    idx_a, idx_b = idx_a[keep], idx_b[keep]

    num = np.sum(vecs[idx_a] * vecs[idx_b], axis=1)
    denom = np.linalg.norm(vecs[idx_a], axis=1) * np.linalg.norm(vecs[idx_b], axis=1)
    sims = np.divide(num, denom, out=np.zeros_like(num), where=denom > 0)

    return pd.DataFrame({
        'protein_a': ids_sub[idx_a],
        'protein_b': ids_sub[idx_b],
        'similarity': sims,
    })

baseline_rows = []
for species in sorted(set(groups_deduped)):
    mask = (groups_deduped == species) & (y_deduped == 0)
    X_sub = X_deduped_kmer[mask]
    ids_sub = X_sub.index
    df = sampled_pairwise_sim(X_sub, ids_sub, n_pairs=2000)
    df['species'] = species.split('_')[0]
    baseline_rows.append(df)

baseline_neg_sim = pd.concat(baseline_rows, ignore_index=True)
baseline_neg_sim.sort_values('similarity', ascending=False).head(10)

#### Cosine similarity distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, (df, title) in zip(axes, [
    (within_species_pos_sim, 'Positive-positive (within species)'),
    (baseline_neg_sim, 'Negative-negative (within species, sampled)'),
]):
    for species, grp in df.groupby('species'):
        ax.hist(grp['similarity'], bins=30, alpha=0.2, label=species)
    ax.set_title(title)
    ax.set_xlabel('Cosine similarity')

axes[0].set_ylabel('Count')
axes[0].legend()
plt.tight_layout()
plt.show()